# Silver: Clean & Conform Population Data

Default lakehouse: `silver_lh`. Attaches `bronze_lh` read-only.
Dedupes on business key `(entity, year)` keeping the latest `batch_id`, casts types, joins continent/WHO region dimensions.
No PII masking — this dataset has no individual-level records (see `specs/design.md`).

In [1]:
try:
    import notebookutils  # noqa: F401  (only importable inside Fabric)
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if not IN_FABRIC:
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession
    builder = (
        SparkSession.builder.appName("urbanisation_prospects_silver")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

DEFAULT_LAKEHOUSE = "silver_lh"
BRONZE_LAKEHOUSE = "bronze_lh"
# Paths are relative to this notebook's own directory (Jupyter's default cwd), i.e. repo_root/notebooks/
LOCAL_LAKEHOUSE_ROOT = "../lakehouse"


def table_path(lakehouse: str, table: str) -> str:
    if IN_FABRIC:
        mount = "default" if lakehouse == DEFAULT_LAKEHOUSE else lakehouse
        return f"/lakehouse/{mount}/Tables/{table}"
    return f"{LOCAL_LAKEHOUSE_ROOT}/{lakehouse}/Tables/{table}"


def read_delta(lakehouse: str, table: str):
    return spark.read.format("delta").load(table_path(lakehouse, table))


def write_delta(lakehouse: str, table: str, df, mode: str = "overwrite"):
    (
        df.write.format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .save(table_path(lakehouse, table))
    )

In [2]:
from pyspark.sql import Window
from pyspark.sql import functions as F


def latest_batch(df, key_cols):
    """Keep only the most recent batch_id per business key (last write wins)."""
    w = Window.partitionBy(*key_cols).orderBy(F.col("batch_id").desc())
    return (
        df.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


def latest_per_entity(df, value_col_alias):
    """Dimension lookups (continent/WHO region) are only keyed by Entity here — take the most recent Year's mapping per entity."""
    w = Window.partitionBy("entity").orderBy(F.col("year").cast("int").desc())
    return (
        df.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "year")
    )

In [3]:
URBAN_COL = "Urban population 1950-2050 (UN World Urbanization Prospects 2018)"
RURAL_COL = "Rural population 1950-2050 (UN World Urbanization Prospects 2018)"

pop_latest = latest_batch(
    read_delta(BRONZE_LAKEHOUSE, "raw_urban_rural_population"), ["Entity", "Year"]
)

population = (
    pop_latest.select(
        F.col("Entity").alias("entity"),
        F.col("Year").cast("int").alias("year"),
        F.col(URBAN_COL).cast("long").alias("urban_population"),
        F.col(RURAL_COL).cast("long").alias("rural_population"),
    )
    .withColumn("is_urban_estimated", F.col("urban_population").isNull())
    .withColumn("is_rural_estimated", F.col("rural_population").isNull())
    .withColumn(
        "total_population",
        F.when(
            F.col("urban_population").isNotNull() | F.col("rural_population").isNotNull(),
            F.coalesce(F.col("urban_population"), F.lit(0))
            + F.coalesce(F.col("rural_population"), F.lit(0)),
        ).otherwise(F.lit(None).cast("long")),
    )
)

In [4]:
continents = latest_per_entity(
    latest_batch(read_delta(BRONZE_LAKEHOUSE, "raw_countries_continents"), ["Entity", "Year"])
    .select(
        F.col("Entity").alias("entity"),
        F.col("Year").alias("year"),
        F.col("Countries Continents").alias("continent"),
    ),
    "continent",
)

who_regions = latest_per_entity(
    latest_batch(read_delta(BRONZE_LAKEHOUSE, "raw_who_regions"), ["Entity", "Year"])
    .select(
        F.col("Entity").alias("entity"),
        F.col("Year").alias("year"),
        F.col("WHO region").alias("who_region"),
    ),
    "who_region",
)

silver_df = (
    population.join(continents, on="entity", how="left")
    .join(who_regions, on="entity", how="left")
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

write_delta(DEFAULT_LAKEHOUSE, "population_urbanization", silver_df, mode="overwrite")
print(f"Silver population_urbanization rows: {silver_df.count()}")

Silver population_urbanization rows: 27573
